# Predicting the Sale Price of Bulldozers using ML

In this notebook, we're going to go through an example machine learning
project with the goal of predicting the sale price of bulldozers.

## 1. Problem Definition

> How well can we predict the future sale price of a bulldozer, given
its characteristics and previous examples of how much similar bulldozer
have been sold for?

## 2. Data

The data is downloaded from the Kaggle Bluebook for Bulldozers competition.

There are 3 main datasets:

* Train.csv is the training set, which contains data through the end of 2011.
* Valid.csv is the validation set, which contains data from January 1, 2012-
April 30, 2012. We make predictions on this set throughout the majority of the
competition. Our score on this set is used to create the public leaderboard.
* Test.csv is the test set, which won't be released until the last week of the
competition. It contains data from May 1, 2012 - November 2012. We calculate
score on the test set, determine our final rank for the competition.


## 3. Evaluation

The evaluation metric fro this competition is the RMSLE(root mean squared log
error) between the actual and predicted auction prices.

**Note**: The goal for most regression evaluation metrics is to minimize the error.
For example, our goal for this project will be to build a machine learning model
which minimises RMSLE.

## 4. Features

Kaggle provides a data dictionary detailing all of the features of the dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Bulldozer Price Prediction/bluebook-for-bulldozers/TrainAndValid.csv', low_memory=False)

In [ ]:
df.info()

In [ ]:
df.isna().sum(), df.isna().sum().size

In [ ]:
df.columns

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(df.saledate[:1000], df.SalePrice[:1000])

In [ ]:
df["saledate"][:1000]

In [ ]:
df["SalePrice"].plot.hist(bins=20, edgecolor="black")

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Bulldozer Price Prediction/bluebook-for-bulldozers/TrainAndValid.csv',
                 low_memory=False,
                 parse_dates=["saledate"])

In [ ]:
df.saledate[:1000]

In [ ]:
df.saledate.dtype

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df.saledate[:1000], df.SalePrice[:1000], alpha=0.5)
# ax.scatter(df.saledate[:1000], df.ModelID[:1000], marker="*", alpha=0.5)

In [ ]:
df.head().T

### Sort DataFrame by saledate

When working with time series data, it's a good idea to sort it by date.

In [ ]:
# Sort DataFrame in date order

df.sort_values(by=["saledate"], inplace=True, ascending=True)

In [ ]:
df.saledate.head(20)

In [ ]:
df.head().T

In [ ]:
# Make a copy
df_temp = df.copy()

In [ ]:
df_temp.saledate[:1], df_temp.saledate[:1].dt.year, df_temp.saledate[:1].dt.month


In [ ]:
df_temp.saledate[:1], df_temp.saledate[:1].dt.day

### Add datetime parameters for saledate column

In [ ]:
df_temp["saleYear"] = df_temp.saledate.dt.year
df_temp["saleMonth"] = df_temp.saledate.dt.month
df_temp["saleDay"] = df_temp.saledate.dt.day
df_temp["saleDayOfWeek"] = df_temp.saledate.dt.dayofweek
df_temp["saleDayOfYear"] = df_temp.saledate.dt.dayofyear

In [ ]:
df_temp.head().T

In [ ]:
# Now we've enriched our DataFrame with date time features, we can remove "saledate"

df_temp.drop("saledate", axis=1, inplace=True)

In [ ]:
# Check the values of different columns

df_temp.state.value_counts()

## 5. Modeling

We'we done enough EDA(we could always do more) but let's start to do some model - driven EDA.

In [ ]:
# Let's build a machine learning model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_jobs=-1,
                              random_state=42)


In [ ]:
df_temp.info()

In [ ]:
df_temp.isna().sum()

### Convert string into categories

One way we can turn all of our data into numbers is by converting them into pandas categories.

In [ ]:
df_temp.UsageBand.dtype

In [ ]:
pd.api.types.is_object_dtype(df_temp["UsageBand"])

In [ ]:
 # Find the columns which contain strings
for label, content in df_temp.items():
  if pd.api.types.is_object_dtype(content):
    print(label)

In [ ]:
# If yor're wondering what df.items() does, here's an example
random_dict = {"key1": "hello",
               "key2": "world!"}

for key, value in random_dict.items():
  print(f"this is a key: {key}",
        f"this is a value: {value}")

In [ ]:
for label, content in df_temp.items():
  if pd.api.types.is_object_dtype(content):
    df_temp[label] = content.astype("category").cat.as_ordered()

In [ ]:
df_temp.info()

In [ ]:
df_temp.state.cat.categories


In [ ]:
df_temp.state.value_counts()

In [ ]:
df_temp.state.cat.codes

*Thanks* to pandas Categories we now have a way to access all of our data
in form of numbers.

But we still have a bunch of missing data...

In [ ]:
# Check missing data
df_temp.isnull().sum()/len(df_temp)

### Save perprocessed data

In [ ]:
# Export currnet temp dataframe
df_temp.to_csv("/content/drive/MyDrive/Bulldozer Price Prediction/bluebook-for-bulldozers/train_tmp.csv",
               index=False)

In [ ]:
# Import Preprocessed Data
df_temp.head().T

In [ ]:
# Import preprocessed data
dr_temp = pd.read_csv("/content/drive/MyDrive/Bulldozer Price Prediction/bluebook-for-bulldozers/train_tmp.csv",
                      low_memory = False)
df_temp.head().T

In [ ]:
df_temp.isna().sum()

### Fill missing values

#### Fill numberical missing values first

In [ ]:
for label, content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    print(label)

In [ ]:
# Check for which numeric columns have null values
for label, content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    if pd.isnull(content).sum():
      print(label)

In [ ]:
# Fill numeric rows with medium
for label,content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    if pd.isnull(content).sum():
      # Add a binary column which tells us if the data was missing or not
      df_temp[label+"_is_missing"] = pd.isnull(content)
      # Fill missing numeric values with median
      df_temp[label] = content.fillna(content.median(), inplace=True)

In [ ]:
df_temp.auctioneerID.isna().sum()

In [ ]:
df_temp.auctioneerID.median();

In [ ]:
hundreds = np.full((1000,), 100)
hundreds_billion = np.append(hundreds, 1000000000)
np.mean(hundreds), np.mean(hundreds_billion), np.median(hundreds), np.median(hundreds_billion)

In [ ]:
for label, content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    if pd.isnull(content).sum():
      print(label)

In [ ]:
df_temp.auctioneerID_is_missing.value_counts()

In [ ]:
df_temp.isna().sum()

In [ ]:
df_temp.auctioneerID_is_missing

In [ ]:
df_temp.auctioneerID_is_missing.value_counts()

In [ ]:
for label, content in df_temp.items():
  if pd.api.types.is_numeric_dtype(content):
    if pd.isnull(content).sum():
      print(f"Column Name : {label}, Missing Value : True")
    else:
      print(f"Column Name : {label}, Missing Value : False")

### Filling and turning categorical variables into numbers

In [ ]:
len = 0
for label, content in df_temp.items():
  if not pd.api.types.is_numeric_dtype(content):
    print(f"Column Name : {label} | Column dtype : {df_temp[label].dtype.name}")
    len += 1
print(len)

In [ ]:
# 1. Create a dictionary to store column to category values (e.g we turn our category types into numbers but we keep a record so we can go back)
column_to_category_dict = {}

# 2. Turn categorical variables into numbers
for label, content in df_temp.items():

  # 3. Check Column's which aren't numberic
  if not pd.api.types.is_numeric_dtype(content):

    # 4. Add binary column to indicate wheater sample had missing value
    df_temp[label+"is_missing"] = pd.isnull(content).astype(int)

    # 5. Ensure content is categorical and get its category codes
    content_categories = pd.Categorical(content)
    content_category_codes = content_categories.codes + 1

    # 6. Add column key to dictionary with code: category mapping per column
    column_to_category_dict[label] = dict(zip(content_category_codes, content_categories))

    # 7. Set the column to the numerical values(the category code value)
    df_temp[label] = content_category_codes

In [ ]:
df_temp.head(10).T

In [ ]:
pd.Categorical(df_temp["state"]).codes + 1

In [ ]:
df_temp.info()

In [ ]:
# Check the first 10 state column values
for key, value in sorted(column_to_category_dict["state"].items())[:10]:
  print(f"{key} -> {value}")

In [ ]:
df_temp.isna().sum()

Now that all of data is numeric as well as our dataframe has no missing values,
we should be able to build a machine learning model.

In [ ]:
df_temp.head()

In [ ]:
%%time
# Instantiate model
model = RandomForestRegressor(n_jobs=-1,
                              random_state=42)
# Fit the model
model.fit(df_temp.drop("SalePrice", axis=1), df_temp["SalePrice"])